IMDB dataset having 50K movie reviews for natural language processing or Text analytics.
This is a dataset for binary sentiment classification containing substantially more data than previous benchmark datasets. We provide a set of 25,000 highly polar movie reviews for training and 25,000 for testing. So, predict the number of positive and negative reviews using either classification or deep learning algorithms.

In [45]:
import os
import time
import kagglehub
import pandas as pd
import numpy as np
import torch
from torch.optim import Adam
from torchinfo import summary
from textblob import TextBlob
import re
import string
from multiprocessing import Pool
from tqdm.auto import tqdm

In [46]:
torch.set_num_threads(10)
torch.manual_seed(42)

In [47]:
# Download latest version
datasetPath = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews", output_dir = "dataset/")

datasetPath = os.path.join(datasetPath, "IMDB Dataset.csv")

print("Path to dataset files:", datasetPath)

Path to dataset files: dataset/IMDB Dataset.csv


In [48]:
rawDF = pd.read_csv(datasetPath, encoding= "latin1", engine= "python", on_bad_lines="skip")
rawDF

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative
49999,No one expects the Star Trek movies to be high...,negative


In [49]:
rawDF.info()

<class 'pandas.DataFrame'>
RangeIndex: 50001 entries, 0 to 50000
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     50001 non-null  str  
 1   sentiment  50001 non-null  str  
dtypes: str(2)
memory usage: 781.4 KB


In [50]:
rawDF.describe()

,review,sentiment
count,50001,50001
unique,49583,2
top,Loved today's show!!! It was a variety and not...,positive
freq,5,25001


# Lower casing

In [51]:
randInt = np.random.randint(0, 50000)
display(rawDF.sample(5, random_state= randInt))
rawDF["review"] = rawDF["review"].str.lower()
rawDF.sample(5, random_state= randInt)

,review,sentiment
46972,"The most generic, surface-level biography you ...",negative
29461,...I saw this movie when it first came out in ...,positive
22652,This wasn't what i wanted to see. I bought thi...,negative
39164,Taking a break from his escapist run in the ea...,positive
7536,"This ""clever"" film was originally a Japanese f...",negative


,review,sentiment
46972,"the most generic, surface-level biography you ...",negative
29461,...i saw this movie when it first came out in ...,positive
22652,this wasn't what i wanted to see. i bought thi...,negative
39164,taking a break from his escapist run in the ea...,positive
7536,"this ""clever"" film was originally a japanese f...",negative


# Remove HTML tags

In [52]:
test = """<html>
<head>
    <title>My Website</title>
</head>
<body>
    <h1>Welcome to My Website</h1>
    <p>This is a paragraph.</p>
    <div>
        <p>I am learning web scraping.</p>
    </div>
</body>
</html>"""

print("Before:")
print(test)


def removeHtml(text):
    htmlPattern = re.compile(r"<.*?>")
    return htmlPattern.sub("", text)

print("\n\nAfter:")
print(removeHtml(test))

Before:
<html>
<head>
    <title>My Website</title>
</head>
<body>
    <h1>Welcome to My Website</h1>
    <p>This is a paragraph.</p>
    <div>
        <p>I am learning web scraping.</p>
    </div>
</body>
</html>


After:


    My Website


    Welcome to My Website
    This is a paragraph.
    
        I am learning web scraping.
    




In [53]:
display(rawDF)

rawDF["review"] = rawDF["review"].apply(removeHtml)

rawDF

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive
...,...,...
49996,"bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,i am a catholic taught in parochial elementary...,negative
49998,i'm going to have to disagree with the previou...,negative
49999,no one expects the star trek movies to be high...,negative


,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. the filming tec...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive
...,...,...
49996,"bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,i am a catholic taught in parochial elementary...,negative
49998,i'm going to have to disagree with the previou...,negative
49999,no one expects the star trek movies to be high...,negative


# Remove links

In [54]:
test = """Check http://example.com, and this is my note https://example.com, more www.example.com example.com now https://example.com/search?q=python sdf sdfsdfsd fsd fsd fsdf sdf sdfsd fsd fsdf
sdf sdf sdf sdf sdf sdf sdf sdfsd fsdf fasds cv fgsdc fasdzxcazzxc httx cz https fsd http zdfz  www"""

print("Before:")
print(test)

def removeLinks(text):
    pattern = re.compile(r"https?://\S+|www\.\S+|\S+\.com")
    return pattern.sub("", text)


print("\n\nAfter:")
print(removeLinks(test))

Before:
Check http://example.com, and this is my note https://example.com, more www.example.com example.com now https://example.com/search?q=python sdf sdfsdfsd fsd fsd fsdf sdf sdfsd fsd fsdf
sdf sdf sdf sdf sdf sdf sdf sdfsd fsdf fasds cv fgsdc fasdzxcazzxc httx cz https fsd http zdfz  www


After:
Check  and this is my note  more   now  sdf sdfsdfsd fsd fsd fsdf sdf sdfsd fsd fsdf
sdf sdf sdf sdf sdf sdf sdf sdfsd fsdf fasds cv fgsdc fasdzxcazzxc httx cz https fsd http zdfz  www


In [55]:
rawDF["review"] = rawDF["review"].apply(removeLinks)
rawDF

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. the filming tec...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive
...,...,...
49996,"bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,i am a catholic taught in parochial elementary...,negative
49998,i'm going to have to disagree with the previou...,negative
49999,no one expects the star trek movies to be high...,negative


# Remove punctuation

In [56]:
punctuation = string.punctuation
punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [57]:
def punctuationRemove(text):
    return text.translate(str.maketrans(" ", " ", punctuation))

In [58]:
test = """Yesterday, I visited the library (which was surprisingly quiet!) to collect some books, articles, and research papers.
I found an interesting book titled "The Art of Programming" — it covered Python, JavaScript, C++, and more. 
Did you know that programming can be both challenging and enjoyable? Well, it can! For example: writing `print("Hello, World!")` is simple; debugging a complex application, however, is another story. The instructor said, "Practice every day, don't give up, and always ask questions." I also found several useful websites: [https://example.com](https://example.com?utm_source=chatgpt.com), www\.google.com, and example.org. Some students prefer online learning; others prefer traditional classrooms. Either way, learning requires patience, consistency, and curiosity. So, what's next? A new project? An exciting experiment? Or perhaps... just a cup of coffee before getting started! Finally, remember this: "Success isn't achieved overnight" — it takes time, effort, discipline, and a little bit of creativity."""

print(test)

test = punctuationRemove(test)
print("\n\n")
print(test)

Yesterday, I visited the library (which was surprisingly quiet!) to collect some books, articles, and research papers.
I found an interesting book titled "The Art of Programming" — it covered Python, JavaScript, C++, and more. 
Did you know that programming can be both challenging and enjoyable? Well, it can! For example: writing `print("Hello, World!")` is simple; debugging a complex application, however, is another story. The instructor said, "Practice every day, don't give up, and always ask questions." I also found several useful websites: [https://example.com](https://example.com?utm_source=chatgpt.com), www\.google.com, and example.org. Some students prefer online learning; others prefer traditional classrooms. Either way, learning requires patience, consistency, and curiosity. So, what's next? A new project? An exciting experiment? Or perhaps... just a cup of coffee before getting started! Finally, remember this: "Success isn't achieved overnight" — it takes time, effort, discip

<>:3: SyntaxWarning: invalid escape sequence '\.'
<>:3: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_11212/2091990673.py:3: SyntaxWarning: invalid escape sequence '\.'
  Did you know that programming can be both challenging and enjoyable? Well, it can! For example: writing `print("Hello, World!")` is simple; debugging a complex application, however, is another story. The instructor said, "Practice every day, don't give up, and always ask questions." I also found several useful websites: [https://example.com](https://example.com?utm_source=chatgpt.com), www\.google.com, and example.org. Some students prefer online learning; others prefer traditional classrooms. Either way, learning requires patience, consistency, and curiosity. So, what's next? A new project? An exciting experiment? Or perhaps... just a cup of coffee before getting started! Finally, remember this: "Success isn't achieved overnight" — it takes time, effort, discipline, and a little bit of creativity."""


In [59]:
display(rawDF)

rawDF["review"] = rawDF["review"].apply(punctuationRemove)

rawDF

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. the filming tec...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive
...,...,...
49996,"bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,i am a catholic taught in parochial elementary...,negative
49998,i'm going to have to disagree with the previou...,negative
49999,no one expects the star trek movies to be high...,negative


,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production the filming tech...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive
...,...,...
49996,bad plot bad dialogue bad acting idiotic direc...,negative
49997,i am a catholic taught in parochial elementary...,negative
49998,im going to have to disagree with the previous...,negative
49999,no one expects the star trek movies to be high...,negative


# Handle spacial words

In [60]:
chat_words = {
    "A3": "Anytime, Anywhere, Anyplace",
    "ADIH": "Another Day In Hell",
    "AFK": "Away From Keyboard",
    "AFAIK": "As Far As I Know",
    "ASAP": "As Soon As Possible",
    "ATK": "At The Keyboard",
    "ATM": "At The Moment",
    "BAE": "Before Anyone Else",
    "BAK": "Back At Keyboard",
    "BBL": "Be Back Later",
    "BBS": "Be Back Soon",
    "BFN": "Bye For Now",
    "B4N": "Bye For Now",
    "BRB": "Be Right Back",
    "BRUH": "Bro",
    "BRT": "Be Right There",
    "BSAAW": "Big Smile And A Wink",
    "BTW": "By The Way",
    "BWL": "Bursting With Laughter",
    "CSL": "Cant Stop Laughing",
    "CU": "See You",
    "CUL8R": "See You Later",
    "CYA": "See You",
    "DM": "Direct Message",
    "FAQ": "Frequently Asked Questions",
    "FC": "Fingers Crossed",
    "FIMH": "Forever In My Heart",
    "FOMO": "Fear Of Missing Out",
    "FR": "For Real",
    "FWIW": "For What It's Worth",
    "FYP": "For You Page",
    "FYI": "For Your Information",
    "G9": "Genius",
    "GAL": "Get A Life",
    "GG": "Good Game",
    "GMTA": "Great Minds Think Alike",
    "GN": "Good Night",
    "GOAT": "Greatest Of All Time",
    "GR8": "Great!",
    "HBD": "Happy Birthday",
    "IC": "I See",
    "ICQ": "I Seek You",
    "IDC": "I Dont Care",
    "IDK": "I Don't Know",
    "IFYP": "I Feel Your Pain",
    "ILU": "I Love You",
    "ILY": "I Love You",
    "IMHO": "In My Honest/Humble Opinion",
    "IMU": "I Miss You",
    "IMO": "In My Opinion",
    "IOW": "In Other Words",
    "IRL": "In Real Life",
    "IYKYK": "If You Know, You Know",
    "JK": "Just Kidding",
    "KISS": "Keep It Simple, Stupid",
    "L": "Loss",
    "L8R": "Later",
    "LDR": "Long Distance Relationship",
    "LMK": "Let Me Know",
    "LMAO": "Laughing My A** Off",
    "LOL": "Laughing Out Loud",
    "LOL!": "Laughing Out Loud",
    "LTNS": "Long Time No See",
    "M8": "Mate",
    "MFW": "My Face When",
    "MID": "Mediocre",
    "MRW": "My Reaction When",
    "MTE": "My Thoughts Exactly",
    "NVM": "Never Mind",
    "NRN": "No Reply Necessary",
    "NPC": "Non-Player Character",
    "OIC": "Oh I See",
    "OP": "Overpowered",
    "POV": "Point Of View",
    "PRT": "Party",
    "PRW": "Parents Are Watching",
    "ROFL": "Rolling On The Floor Laughing",
    "ROFLOL": "Rolling On The Floor Laughing Out Loud",
    "ROTFLMAO": "Rolling On The Floor Laughing My A** Off",
    "RN": "Right Now",
    "SK8": "Skate",
    "SUS": "Suspicious",
    "TBH": "To Be Honest",
    "TFW": "That Feeling When",
    "THX": "Thank You",
    "TIME": "Tears In My Eyes",
    "TLDR": "Too Long, Didnt Read",
    "TNTL": "Trying Not To Laugh",
    "TTFN": "Ta-Ta For Now!",
    "TTYL": "Talk To You Later",
    "U": "You",
    "U2": "You Too",
    "U4E": "Yours For Ever",
    "W": "Win",
    "W8": "Wait...",
    "WB": "Welcome Back",
    "WTG": "Way To Go",
    "WUF": "Where Are You From?",
    "WYD": "What You Doing?",
    "WYWH": "Wish You Were Here",
    "ZZZ": "Sleeping, Bored, Tired",
    "OMG": "Oh My God",
    "SMH": "Shaking My Head",
    "NGL": "Not Gonna Lie",
    "TBF": "To Be Fair",
    "ICYMI": "In Case You Missed It",
    "IMO": "In My Opinion",
    "IMHO": "In My Honest Opinion",
    "RN": "Right Now",
    "FR": "For Real",
    "DW": "Don't Worry",
    "ILYSM": "I Love You So Much",
    "OMW": "On My Way",
    "GGWP": "Good Game Well Played",
    "GG": "Good Game",
    "EZ": "Easy",
    "LMAO": "Laughing My Ass Off",
    "LOL": "Laughing Out Loud",
    "AAMOF": "As A Matter Of Fact",
    "AFAIC": "As Far As I'm Concerned",
    "AF": "As Fuck",
    "ASL": "Age, Sex, Location",
    "AWOL": "Absent Without Leave",
    "BC": "Because",
    "BFF": "Best Friends Forever",
    "BFFL": "Best Friends For Life",
    "BTAIM": "Be That As It May",
    "BTS": "Behind The Scenes",
    "DND": "Do Not Disturb",
    "EOD": "End Of Day",
    "ETA": "Estimated Time Of Arrival",
    "F2F": "Face To Face",
    "FTW": "For The Win",
    "G2G": "Got To Go",
    "GTG": "Got To Go",
    "HAND": "Have A Nice Day",
    "HBU": "How About You",
    "HMU": "Hit Me Up",
    "HTH": "Hope This Helps",
    "IAC": "In Any Case",
    "IANAL": "I Am Not A Lawyer",
    "ICBW": "I Could Be Wrong",
    "IDGAF": "I Don't Give A Fuck",
    "IIRC": "If I Recall Correctly",
    "IKR": "I Know, Right",
    "ILY2": "I Love You Too",
    "IMNSHO": "In My Not So Humble Opinion",
    "IRRC": "If I Remember Correctly",
    "ISTG": "I Swear To God",
    "JIC": "Just In Case",
    "JSYK": "Just So You Know",
    "K": "Okay",
    "KK": "Okay",
    "LMFAO": "Laughing My Fucking Ass Off",
    "MB": "My Bad",
    "MSG": "Message",
    "MYOB": "Mind Your Own Business",
    "NBD": "No Big Deal",
    "NP": "No Problem",
    "NGL": "Not Gonna Lie",
    "NOYB": "None Of Your Business",
    "OFC": "Of Course",
    "OMFG": "Oh My Fucking God",
    "OT": "Off Topic",
    "OTW": "On The Way",
    "PFA": "Please Find Attached",
    "PLS": "Please",
    "PLZ": "Please",
    "PM": "Private Message",
    "PPL": "People",
    "QOTD": "Question Of The Day",
    "RN": "Right Now",
    "ROFL": "Rolling On The Floor Laughing",
    "SFW": "Safe For Work",
    "SOML": "Story Of My Life",
    "SRSLY": "Seriously",
    "STFU": "Shut The Fuck Up",
    "TBA": "To Be Announced",
    "TBC": "To Be Confirmed",
    "TIA": "Thanks In Advance",
    "TIL": "Today I Learned",
    "TMI": "Too Much Information",
    "TT": "Trending Topic",
    "TY": "Thank You",
    "TYSM": "Thank You So Much",
    "WBU": "What About You",
    "WTF": "What The Fuck",
    "WTH": "What The Heck",
    "XOXO": "Hugs And Kisses",
    "YW": "You're Welcome",

    # Modern internet slang
    "ATP": "At This Point",
    "BFR": "Be For Real",
    "BOP": "A Really Good Song",
    "CAP": "A Lie",
    "CHEUGY": "Outdated Or Unfashionable",
    "DEAD": "Extremely Amused",
    "DRIP": "Fashionable Style",
    "FIRE": "Really Good Or Excellent",
    "GOATED": "Considered The Greatest",
    "GYAT": "Expression Of Surprise Or Admiration",
    "IYKYK": "If You Know, You Know",
    "LET HIM COOK": "Let Someone Continue What They're Doing",
    "LOWKEY": "Somewhat Or Secretly",
    "HIGHKEY": "Very Much Or Obviously",
    "MAIN CHARACTER": "Someone Acting Like The Center Of Attention",
    "NO CAP": "No Lie",
    "RIZZ": "Charisma Or Romantic Charm",
    "SHEESH": "Expression Of Amazement",
    "SLAY": "To Do Something Exceptionally Well",
    "SUSSY": "Suspicious",
    "VALID": "Acceptable Or Relatable",
    "VIBE": "Mood Or Atmosphere",
    "WILD": "Unexpected Or Shocking",
    "YAP": "Talk Excessively",
    "YAPPER": "Someone Who Talks A Lot",
    "YEET": "To Throw Something Forcefully",
    "ATE": "Did Something Exceptionally Well",
    "ATE AND LEFT NO CRUMBS": "Did Something Perfectly",
    "BET": "Okay Or Agreed",
    "BRB": "Be Right Back",
    "COOKED": "In Serious Trouble",
    "DELULU": "Delusional",
    "GOAT": "Greatest Of All Time",
    "LOCK IN": "Focus Seriously",
    "MID": "Average Or Mediocre",
    "OOMF": "One Of My Followers",
    "Pookie": "A Term Of Endearment",
    "REAL": "Strong Agreement",
    "SKIBIDI": "Internet Slang With Context-Dependent Meaning",
    "TOUCH GRASS": "Spend Time Away From The Internet",
    "UNDERSTOOD THE ASSIGNMENT": "Did Something Exactly Right",
    "WOMP WOMP": "Expression Of Dismissal Or Disappointment",

    # Social media abbreviations
    "AMA": "Ask Me Anything",
    "BTS": "Behind The Scenes",
    "CF": "Close Friends",
    "F4F": "Follow For Follow",
    "FB": "Facebook",
    "FBO": "Facebook Official",
    "GRWM": "Get Ready With Me",
    "IG": "Instagram",
    "IGTV": "Instagram TV",
    "L4L": "Like For Like",
    "OOTD": "Outfit Of The Day",
    "QRT": "Quote Retweet",
    "RT": "Retweet",
    "TBT": "Throwback Thursday",
    "TIL": "Today I Learned",
    "YT": "YouTube",
    "YTTA": "YouTube To The Audience",

    # Gaming abbreviations
    "ADS": "Aim Down Sights",
    "AIMBOT": "Automated Aiming Cheat",
    "AOE": "Area Of Effect",
    "BR": "Battle Royale",
    "CD": "Cooldown",
    "DPS": "Damage Per Second",
    "FPS": "First Person Shooter",
    "HP": "Health Points",
    "HUD": "Heads Up Display",
    "KDA": "Kills Deaths Assists",
    "MMO": "Massively Multiplayer Online",
    "MMORPG": "Massively Multiplayer Online Role Playing Game",
    "MVP": "Most Valuable Player",
    "P2W": "Pay To Win",
    "PVP": "Player Versus Player",
    "PVE": "Player Versus Environment",
    "RPG": "Role Playing Game",
    "RTS": "Real Time Strategy",
    "XP": "Experience Points",
    "AFK FARMING": "Gaining Resources While Away",
}

In [61]:
def expandWords(text):
    output = []

    for word in text.split():
        if word.upper() in chat_words.keys():
            output.append(chat_words[word.upper()].lower())
        else:
            output.append(word)
    return " ".join(output)

In [62]:
test = """OMG WTF you are saying
I am in sleep LOL idk"""
print(test)
print(f"\n\n{expandWords(test)}")

OMG WTF you are saying
I am in sleep LOL idk


oh my god what the fuck you are saying I am in sleep laughing out loud i don't know


In [63]:
display(rawDF)

rawDF["review"] = rawDF["review"].apply(expandWords)

rawDF

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production the filming tech...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive
...,...,...
49996,bad plot bad dialogue bad acting idiotic direc...,negative
49997,i am a catholic taught in parochial elementary...,negative
49998,im going to have to disagree with the previous...,negative
49999,no one expects the star trek movies to be high...,negative


,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production the filming tech...,positive
2,i thought this was a wonderful way to spend te...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the tears in my eyes of...,positive
...,...,...
49996,bad plot bad dialogue bad acting idiotic direc...,negative
49997,i am a catholic taught in parochial elementary...,negative
49998,im going to have to disagree with the previous...,negative
49999,no one expects the star trek movies to be high...,negative


# Handle Misspelling

In [64]:
demo = "Smiel, hey, helloo, hoow are you doinng totally wrrong"
print(demo)

def makeCorrect(text):
    if not isinstance(text, str):
        return text

    return str(TextBlob(text).correct())

print(makeCorrect(demo))

Smiel, hey, helloo, hoow are you doinng totally wrrong
Smile, hey, hello, how are you doing totally wrong


In [65]:
"""
sample = rawDF["review"].sample(
    n=min(1000, len(rawDF)),
    random_state=42
).tolist()

start = time.perf_counter()

with Pool(processes=11) as pool:
    results = list(
        tqdm(
            pool.imap(makeCorrect, sample, chunksize=50),
            total=len(sample)
        )
    )

elapsed = time.perf_counter() - start

estimated_time = elapsed * (len(rawDF) / len(sample))

print(f"Time for 1,000 reviews: {elapsed:.2f} seconds")
print(f"Estimated total time: {estimated_time / 60:.2f} minutes")
"""

'\nsample = rawDF["review"].sample(\n    n=min(1000, len(rawDF)),\n    random_state=42\n).tolist()\n\nstart = time.perf_counter()\n\nwith Pool(processes=11) as pool:\n    results = list(\n        tqdm(\n            pool.imap(makeCorrect, sample, chunksize=50),\n            total=len(sample)\n        )\n    )\n\nelapsed = time.perf_counter() - start\n\nestimated_time = elapsed * (len(rawDF) / len(sample))\n\nprint(f"Time for 1,000 reviews: {elapsed:.2f} seconds")\nprint(f"Estimated total time: {estimated_time / 60:.2f} minutes")\n'

In [66]:
"""
display(rawDF)

with Pool(processes=11) as pool:
    rawDF["review"] = list(
        tqdm(
            pool.imap(
                makeCorrect,
                rawDF["review"],
                chunksize=100
            ),
            total=len(rawDF)
        )
    )

rawDF
"""

'\ndisplay(rawDF)\n\nwith Pool(processes=11) as pool:\n    rawDF["review"] = list(\n        tqdm(\n            pool.imap(\n                makeCorrect,\n                rawDF["review"],\n                chunksize=100\n            ),\n            total=len(rawDF)\n        )\n    )\n\nrawDF\n'